# Cloud Composer 3: Concurrency Bottleneck Proof & Deferrable Operator Resolution

This notebook provides a fully automated, end-to-end workflow to **replicate, prove, and resolve** Cloud Composer 3 task scheduling and worker queue limitations.

## Background

When executing a high volume of lightweight tasks (such as API polling, sensors, or Airbyte syncs), Cloud Composer environments experience a severe bottleneck:

1. **Worker concurrency saturation:** Each worker can only run `[celery]worker_concurrency` tasks simultaneously. With default settings, this is ~6-12 tasks per worker.
2. **Autoscaling trap:** Because lightweight tasks consume near-zero CPU, the GKE autoscaler fails to spin up additional worker pods, leaving tasks stuck in `Queued` state.
3. **Hidden scheduler cap:** The `[core]parallelism` setting (default: 32) silently caps how many tasks the scheduler can place into the executor queue globally, regardless of worker capacity.

## Resolution: Deferrable Operators

The GCP best practice is to use **deferrable operators** (`deferrable=True`). These split task execution into three phases:
1. **Initialize:** Worker starts the task, registers a trigger, and **immediately yields the worker slot**.
2. **Defer:** The task moves to the **Triggerer** process (async `asyncio` event loop) which can handle hundreds of concurrent tasks with near-zero resource overhead.
3. **Resume:** When the trigger fires, a worker runs a lightweight callback to complete the task.

## Notebook Stages
1. **Stage 1:** Setup & Parameterization
2. **Stage 2:** Environment Provisioning
3. **Stage 3:** DAG Deployment (synchronous + deferrable)
4. **Stage 4:** Baseline Experiment (replicating the bottleneck with synchronous tasks)
5. **Stage 5:** Environment Recovery (cleaning up after the baseline lockup)
6. **Stage 6:** Deferrable Experiment (proving the solution)
7. **Stage 7:** Results Comparison
8. **Stage 8:** Teardown & Cleanup

### References
- [Cloud Composer 3 - Deferrable Operators](https://docs.cloud.google.com/composer/docs/composer-3/use-deferrable-operators)
- [Cloud Composer 3 - Optimize Environments](https://docs.cloud.google.com/composer/docs/composer-3/optimize-environments)
- [Cloud Composer 3 - Override Airflow Configurations](https://docs.cloud.google.com/composer/docs/composer-3/override-airflow-configurations)
- [Cloud Composer 3 - Update Environments (restart behavior)](https://docs.cloud.google.com/composer/docs/composer-3/update-environments)
- [Apache Airflow - Deferring Tasks](https://airflow.apache.org/docs/apache-airflow/stable/concepts/deferring.html)

---
## Stage 1: Setup & Parameterization

Configure your Google Cloud parameters below. Ensure your active `gcloud` session has editor/owner privileges on the specified project.

In [ ]:
import subprocess
import json
import time
import textwrap

# ============================================================
# USER CONFIGURATION - Customize these for your GCP account
# ============================================================
PROJECT_ID = "your-project-id"          # Your GCP project ID
REGION = "us-central1"                  # Region for the Composer environment
ENVIRONMENT_NAME = "stress-test-composer"  # Name for the temporary environment

# Task count for the stress test (simulates 70 concurrent lightweight API/sensor tasks)
TASK_COUNT = 70
# Sleep duration in seconds (simulates lightweight API polling)
SLEEP_DURATION = 30

print(f"Project:     {PROJECT_ID}")
print(f"Region:      {REGION}")
print(f"Environment: {ENVIRONMENT_NAME}")
print(f"Task Count:  {TASK_COUNT}")
print(f"Sleep/Wait:  {SLEEP_DURATION}s")

In [ ]:
# Helper functions used throughout the notebook

def run_gcloud(*args, check=True, capture=True):
    """Run a gcloud command and return stdout."""
    cmd = ["gcloud"] + list(args) + [f"--project={PROJECT_ID}"]
    result = subprocess.run(cmd, capture_output=capture, text=True, check=check)
    return result.stdout if capture else None

def run_airflow_cli(*args):
    """Run an Airflow CLI command via gcloud composer environments run."""
    cmd = [
        "gcloud", "composer", "environments", "run", ENVIRONMENT_NAME,
        f"--location={REGION}", f"--project={PROJECT_ID}"
    ] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout

def get_task_state_counts(dag_id, run_id):
    """Query task states for a DAG run and return state counts."""
    output = run_airflow_cli("tasks", "states-for-dag-run", "--", dag_id, run_id, "--output=json")
    # Extract JSON from output (skip gcloud stderr lines)
    for line in output.splitlines():
        line = line.strip()
        if line.startswith("["):
            tasks = json.loads(line)
            break
    else:
        # Try parsing the whole output
        tasks = json.loads(output)
    states = {}
    for task in tasks:
        state = task.get("state", "unknown")
        states[state] = states.get(state, 0) + 1
    return states, tasks

def trigger_dag(dag_id):
    """Trigger a DAG and return the run_id."""
    output = run_airflow_cli("dags", "trigger", "--", dag_id)
    # Parse run_id from the trigger output
    for line in output.splitlines():
        if "manual__" in line:
            # Handle both table format (pipe-delimited) and plain output
            if "|" in line:
                for col in line.split("|"):
                    col = col.strip()
                    if col.startswith("manual__"):
                        return col
            else:
                import re
                match = re.search(r'(manual__\S+)', line)
                if match:
                    return match.group(1)
    raise ValueError(f"Could not extract DAG Run ID from trigger output:\n{output}")

print("Helper functions loaded.")

---
## Stage 2: Provision Temporary Composer 3 Environment

Creates a **Small** preset environment. This takes approximately 15-25 minutes.

The Small preset provides:
- Workers: 1-3 pods, 0.5 vCPU / 2.5 GB each (auto-calculated `worker_concurrency` ~6)
- Scheduler: 1 instance
- Triggerer: 1 instance (required for deferrable operators)

> **Note:** If you already have an environment you want to test against, skip this cell and set `ENVIRONMENT_NAME` to your existing environment name.

In [ ]:
print(f"Creating Composer 3 environment '{ENVIRONMENT_NAME}'...")
print("This typically takes 15-25 minutes.\n")

create_cmd = [
    "gcloud", "composer", "environments", "create", ENVIRONMENT_NAME,
    f"--location={REGION}",
    f"--project={PROJECT_ID}",
    "--environment-size=small"
]

process = subprocess.Popen(create_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    print("\nComposer environment provisioned successfully.")
else:
    print("\nEnvironment creation failed. Check the output above.")

---
## Stage 3: Deploy Stress Test DAGs

We deploy **two** DAGs to the environment's GCS DAG bucket:

1. **`synchronous_stress_test`** - 70 parallel `PythonOperator` tasks with `time.sleep(30)`. These occupy a worker slot for the full 30 seconds. This replicates the bottleneck.

2. **`deferrable_stress_test`** - 70 parallel `TimeDeltaSensor` tasks with `deferrable=True`. Each task initializes on a worker, immediately defers to the Triggerer, and frees its worker slot. This proves the solution.

In [ ]:
# Retrieve DAG bucket path
describe_output = run_gcloud(
    "composer", "environments", "describe", ENVIRONMENT_NAME,
    f"--location={REGION}", "--format=json"
)
env_data = json.loads(describe_output)
dag_bucket = env_data["config"]["dagGcsPrefix"]
print(f"DAG Bucket: {dag_bucket}\n")

# DAG 1: Synchronous stress test (replicates the bottleneck)
sync_dag = textwrap.dedent(f"""\
    import datetime
    import time
    from airflow import DAG
    from airflow.operators.python import PythonOperator

    def sleep_fn(task_num):
        print(f"Starting lightweight sleep task #{{task_num}}...")
        time.sleep({SLEEP_DURATION})
        print(f"Finished sleep task #{{task_num}}.")

    with DAG(
        dag_id="synchronous_stress_test",
        schedule=None,
        start_date=datetime.datetime(2023, 1, 1),
        catchup=False,
        max_active_tasks={TASK_COUNT},
        tags=["stress-test"],
    ) as dag:
        for i in range(1, {TASK_COUNT + 1}):
            PythonOperator(
                task_id=f"sleep_task_{{i}}",
                python_callable=sleep_fn,
                op_kwargs={{"task_num": i}},
            )
""")

# DAG 2: Deferrable stress test (proves the solution)
defer_dag = textwrap.dedent(f"""\
    import datetime
    from airflow import DAG
    from airflow.sensors.time_delta import TimeDeltaSensor

    with DAG(
        dag_id="deferrable_stress_test",
        schedule=None,
        start_date=datetime.datetime(2023, 1, 1),
        catchup=False,
        max_active_tasks={TASK_COUNT},
        tags=["stress-test"],
    ) as dag:
        for i in range(1, {TASK_COUNT + 1}):
            TimeDeltaSensor(
                task_id=f"deferrable_task_{{i}}",
                delta=datetime.timedelta(seconds={SLEEP_DURATION}),
                deferrable=True,
            )
""")

# Write locally and upload
for filename, content in [("synchronous_stress_test.py", sync_dag), ("deferrable_stress_test.py", defer_dag)]:
    with open(filename, "w") as f:
        f.write(content)
    subprocess.run(["gsutil", "cp", filename, f"{dag_bucket}/"], check=True)
    print(f"Uploaded {filename}")

print("\nBoth DAGs deployed. Waiting 60s for DAG processor to parse them...")
time.sleep(60)
print("Done.")

---
## Stage 4: Baseline Experiment (Replicating the Bottleneck)

We apply configuration overrides that expose the scheduling bottleneck:

| Parameter | Value | Effect |
|---|---|---|
| `[core]parallelism` | `150` | Allow scheduler to queue up to 150 tasks |
| `[core]max_active_tasks_per_dag` | `150` | Remove per-DAG task cap |

We deliberately **do not** increase `worker_concurrency` to show what happens when 70 tasks hit a worker with ~6 slots.

### Expected Behavior
- **~6 tasks `running`** (limited by default `worker_concurrency` on 0.5 vCPU)
- **~64 tasks `queued`** (stuck because worker slots are saturated and CPU-based autoscaling won't trigger for `time.sleep()` tasks)
- Tasks will eventually complete in batches of ~6 as each 30-second sleep finishes

> **WARNING:** On Small environments, do NOT set `[celery]worker_concurrency` above ~12.
> Setting it to 23-24 causes an Out-Of-Memory crash loop (23 prefork processes * ~150MB = ~3.45GB, exceeding the 2.5GB worker memory limit), which crashes the database proxy sidecar and locks out the entire environment.
> See the Recovery stage (Stage 5) for the documented recovery procedure.

In [ ]:
print("Applying configuration overrides...")
print("(This restarts all Airflow components and takes 3-5 minutes)\n")

subprocess.run([
    "gcloud", "composer", "environments", "update", ENVIRONMENT_NAME,
    f"--location={REGION}", f"--project={PROJECT_ID}",
    "--update-airflow-configs=core-parallelism=150,core-max_active_tasks_per_dag=150"
], check=True)

print("Configuration applied.\n")
print("Waiting 30s for components to stabilize...")
time.sleep(30)

# Trigger the synchronous stress test
print("Triggering synchronous_stress_test...")
sync_run_id = trigger_dag("synchronous_stress_test")
print(f"DAG Run ID: {sync_run_id}")

print(f"Waiting {SLEEP_DURATION}s for tasks to schedule and hit the bottleneck...")
time.sleep(SLEEP_DURATION)

# Query task states
sync_states, sync_tasks = get_task_state_counts("synchronous_stress_test", sync_run_id)

print("\n" + "=" * 50)
print("BASELINE RESULTS (Synchronous Tasks)")
print("=" * 50)
for state, count in sorted(sync_states.items()):
    print(f"  {state:12s}: {count} tasks")
print(f"  {'TOTAL':12s}: {sum(sync_states.values())} tasks")
print("=" * 50)

queued = sync_states.get("queued", 0)
running = sync_states.get("running", 0)
if queued > 0:
    print(f"\nBOTTLENECK CONFIRMED: {queued} tasks stuck in 'queued' state.")
    print(f"Only {running} tasks could execute concurrently (worker_concurrency limit).")
    print("The remaining tasks must wait for slots to free up.")
else:
    print("\nNote: All tasks may have already completed if the sleep duration was short.")

---
## Stage 5: Environment Recovery

After the baseline experiment, we need to ensure the environment is healthy before running the deferrable test.

### Why Recovery Is Needed

When a synchronous stress test overwhelms the workers, two cascading failures can occur:

1. **Executor desynchronization:** If tasks are manually marked as failed/skipped while workers are frozen, the scheduler's in-memory executor maps (`queued_tasks`, `running_tasks`) may not clear properly. The scheduler then believes all parallelism slots are occupied and refuses to queue new tasks for *any* DAG -- printing `Executor parallelism limit reached. 0 open slots.` every second.

2. **Worker OOM crash loop (if `worker_concurrency` was set too high):** On Small environments (0.5 vCPU / 2.5 GB), setting `worker_concurrency` above ~12 causes OOM kills, which crash the database proxy sidecar container. Workers enter an infinite `Waiting for Celery to start up` loop with `Connection refused` on `localhost:3306`.

### Recovery Procedure

Per [official documentation](https://docs.cloud.google.com/composer/docs/composer-3/update-environments): *"Adding, changing, or deleting Airflow configuration options overrides"* causes **all Airflow components to restart** (schedulers AND workers). This is the only recovery mechanism in Composer 3, since the GKE cluster is in Google's tenant project and users cannot `kubectl` into pods.

The cell below:
1. Pauses both stress test DAGs
2. Waits for any running tasks to drain
3. Verifies `airflow_monitoring` is healthy

In [ ]:
print("=" * 50)
print("ENVIRONMENT RECOVERY")
print("=" * 50)

# Step 1: Pause both stress test DAGs
print("\n[1/3] Pausing stress test DAGs...")
for dag_id in ["synchronous_stress_test", "deferrable_stress_test"]:
    try:
        run_airflow_cli("dags", "pause", "--", dag_id)
        print(f"  Paused {dag_id}")
    except subprocess.CalledProcessError:
        print(f"  {dag_id} already paused or not found")

# Step 2: Wait for running tasks to drain
print("\n[2/3] Waiting 60s for any running tasks to drain...")
time.sleep(60)

# Step 3: Check airflow_monitoring health
print("\n[3/3] Checking airflow_monitoring health...")
try:
    monitoring_output = run_airflow_cli("dags", "list-runs", "--", "airflow_monitoring", "--output", "table")
    # Find the most recent run
    for line in monitoring_output.splitlines():
        if "airflow_monitoring" in line and ("success" in line or "running" in line or "failed" in line):
            state = "success" if "success" in line else ("running" if "running" in line else "failed")
            print(f"  Latest airflow_monitoring run: {state}")
            break
    if "success" in line:
        print("  Environment is HEALTHY.")
    elif "running" in line:
        print("  Environment is RECOVERING (monitoring run in progress).")
    else:
        print("  Environment may still be recovering. Check the Airflow UI.")
        print("  If 'airflow_monitoring' continues failing, run this recovery command:")
        print(f"  gcloud composer environments update {ENVIRONMENT_NAME} \\")
        print(f"    --location={REGION} --project={PROJECT_ID} \\")
        print(f"    --remove-airflow-configs=celery-worker_concurrency")
        print("  This forces a full restart of all Airflow components.")
except subprocess.CalledProcessError as e:
    print(f"  Could not check monitoring health: {e}")

print("\nRecovery check complete.")

---
## Stage 6: Deferrable Experiment (Proving the Solution)

Now we run the deferrable stress test using `TimeDeltaSensor(deferrable=True)`.

### How Deferrable Tasks Differ

| Phase | Synchronous Task | Deferrable Task |
|---|---|---|
| **Start** | Worker picks up task, holds slot for 30s | Worker picks up task, registers trigger, **immediately releases slot** |
| **Wait** | Worker slot occupied (sleeping) | Task moves to **Triggerer** (asyncio event loop, near-zero resources) |
| **Complete** | Worker marks task complete | Triggerer fires event, worker runs lightweight callback |
| **Worker slots needed** | 70 concurrent (impossible with 6 slots) | ~6 momentarily for init, then **0** during the 30s wait |

### Expected Behavior
- All 70 tasks will complete successfully within ~60-90 seconds
- Worker slot usage stays minimal (tasks defer almost instantly)
- Only 1 worker pod needed (no autoscaling required)

In [ ]:
# Unpause the deferrable stress test
print("Unpausing deferrable_stress_test...")
try:
    run_airflow_cli("dags", "unpause", "--", "deferrable_stress_test")
except subprocess.CalledProcessError:
    pass  # Already unpaused

# Trigger the deferrable stress test
print("Triggering deferrable_stress_test...")
defer_run_id = trigger_dag("deferrable_stress_test")
print(f"DAG Run ID: {defer_run_id}")

# Wait for tasks to initialize, defer, and complete
# Deferrable tasks: ~5s init + 30s on triggerer + ~5s callback = ~40s
# But with 70 tasks and ~6 worker slots, init takes ~70/6 * 2s = ~24s
wait_time = SLEEP_DURATION + 60
print(f"Waiting {wait_time}s for all tasks to defer and complete...")
time.sleep(wait_time)

# Query task states
defer_states, defer_tasks = get_task_state_counts("deferrable_stress_test", defer_run_id)

print("\n" + "=" * 50)
print("DEFERRABLE RESULTS")
print("=" * 50)
for state, count in sorted(defer_states.items()):
    print(f"  {state:12s}: {count} tasks")
print(f"  {'TOTAL':12s}: {sum(defer_states.values())} tasks")
print("=" * 50)

success = defer_states.get("success", 0)
if success == TASK_COUNT:
    print(f"\nSOLUTION VALIDATED: All {TASK_COUNT} deferrable tasks completed successfully!")
    print("Tasks deferred to the Triggerer and freed worker slots immediately.")
    print("No autoscaling or worker_concurrency increase needed.")
else:
    remaining = TASK_COUNT - success
    print(f"\n{success}/{TASK_COUNT} tasks completed. {remaining} still in progress.")
    print("If tasks are in 'deferred' state, they are running on the Triggerer.")
    print("Wait a bit longer and re-run this cell to check final results.")

---
## Stage 7: Results Comparison

Side-by-side comparison of the synchronous vs deferrable experiments.

In [ ]:
print("=" * 70)
print("RESULTS COMPARISON: Synchronous vs Deferrable")
print("=" * 70)
print(f"{'Metric':<40s} {'Synchronous':>12s} {'Deferrable':>12s}")
print("-" * 70)
print(f"{'Total Tasks':<40s} {TASK_COUNT:>12d} {TASK_COUNT:>12d}")
print(f"{'Tasks Stuck in Queued':<40s} {sync_states.get('queued', 0):>12d} {defer_states.get('queued', 0):>12d}")
print(f"{'Tasks Running (at snapshot)':<40s} {sync_states.get('running', 0):>12d} {defer_states.get('running', 0):>12d}")
print(f"{'Tasks Succeeded':<40s} {sync_states.get('success', 0):>12d} {defer_states.get('success', 0):>12d}")
print(f"{'Tasks Deferred (on Triggerer)':<40s} {sync_states.get('deferred', 0):>12d} {defer_states.get('deferred', 0):>12d}")
print("-" * 70)
print(f"{'Worker Slots Required':<40s} {f'{TASK_COUNT} (all)':>12s} {'~6 (brief)':>12s}")
print(f"{'Autoscaling Required':<40s} {'Yes':>12s} {'No':>12s}")
print(f"{'Worker OOM Risk':<40s} {'High':>12s} {'None':>12s}")
print("=" * 70)

print("\nCONCLUSION:")
print("Deferrable operators (deferrable=True) resolve the Composer concurrency")
print("bottleneck by offloading lightweight polling/wait tasks to the Triggerer's")
print("asyncio event loop. This eliminates worker slot saturation, removes the")
print("need for CPU-based autoscaling, and prevents OOM crash loops.")
print("\nFor production Airbyte workloads, use AirbyteTriggerSyncOperator")
print("with deferrable=True instead of synchronous or reschedule mode.")
print("\nFor an Airbyte-specific stress test using AirbyteTriggerSyncOperator(deferrable=True),")
print("see dags/airbyte_deferrable_stress_test.py and ARCHITECTURE.md in this repository.")

---
## Stage 8: Teardown & Cleanup

Remove all temporary resources to prevent billing costs.

> **Skip this cell** if you want to keep the environment for further testing.

In [ ]:
print("Cleaning up DAG files from GCS...")
for dag_file in ["synchronous_stress_test.py", "deferrable_stress_test.py"]:
    try:
        subprocess.run(["gsutil", "rm", f"{dag_bucket}/{dag_file}"], check=True)
        print(f"  Removed {dag_file} from GCS")
    except subprocess.CalledProcessError:
        print(f"  {dag_file} not found in GCS (already removed)")

import os
for local_file in ["synchronous_stress_test.py", "deferrable_stress_test.py"]:
    if os.path.exists(local_file):
        os.remove(local_file)

print(f"\nDeleting Composer environment '{ENVIRONMENT_NAME}'...")
print("This takes 5-10 minutes.\n")

delete_cmd = [
    "gcloud", "composer", "environments", "delete", ENVIRONMENT_NAME,
    f"--location={REGION}", f"--project={PROJECT_ID}", "--quiet"
]
process = subprocess.Popen(delete_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    print("\nCleanup complete. Environment deleted.")
else:
    print("\nEnvironment deletion may have failed. Check the GCP Console.")